# Treatment Decision Explorer

This notebook helps analyze treatment decisions for patients based on:
- Patient demographics (subject_id, sex, age)
- Historical treatment effects for similar patients
- Lab value changes before and after treatments

Use this to determine whether to give a specific treatment to a patient based on their characteristics and historical data.

In [ ]:
import polars as pl
import datetime
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set data directory
DATA_DIR = Path("processed")

## 1. Load Data

In [ ]:
# Load all datasets
patients = pl.read_parquet(DATA_DIR / "patients.parquet")
medications = pl.read_parquet(DATA_DIR / "medications.parquet")
labs = pl.read_parquet(DATA_DIR / "labs.parquet")

print(f"Patients: {len(patients):,}")
print(f"Medication records: {len(medications):,}")
print(f"Lab records: {len(labs):,}")

## 2. Patient Selection and Demographics

In [ ]:
# View available patients with their demographics
patient_summary = (
    patients
    .select(["subject_id", "sex", "anchor_age"])
    .head(20)
)
print("Sample patients:")
print(patient_summary)

In [ ]:
# SELECT A PATIENT TO ANALYZE
# Change this to analyze different patients
PATIENT_ID = 10014729  # Example patient ID

# Get patient information
patient_info = patients.filter(pl.col("subject_id") == PATIENT_ID)

if len(patient_info) == 0:
    print(f"Patient {PATIENT_ID} not found!")
else:
    print("Patient Information:")
    print(f"  Subject ID: {PATIENT_ID}")
    print(f"  Sex: {patient_info['sex'][0]}")
    print(f"  Age: {patient_info['anchor_age'][0]}")

## 3. View Available Drugs and Lab Tests

In [ ]:
# Get top drugs by frequency
top_drugs = (
    medications
    .filter(pl.col("drug").is_not_null())
    .group_by("drug")
    .agg(pl.count().alias("count"))
    .sort("count", descending=True)
    .head(20)
)
print("Top 20 drugs by frequency:")
print(top_drugs)

In [ ]:
# Get top lab tests by frequency
top_labs = (
    labs
    .group_by("lab_test")
    .agg(pl.count().alias("count"))
    .sort("count", descending=True)
    .head(20)
)
print("Top 20 lab tests by frequency:")
print(top_labs)

## 4. Analyze Treatment Effects

This section analyzes how a specific drug affects lab values by comparing measurements before and after treatment.

In [ ]:
def analyze_treatment_effect(drug_name: str, lab_test: str, sex_filter: str = None, age_range: tuple = None):
    """
    Analyze treatment effectiveness by comparing lab values before and after medication.
    
    Args:
        drug_name: Drug name (substring match, case-insensitive)
        lab_test: Lab test name (substring match, case-insensitive)
        sex_filter: Filter by sex ('M' or 'F')
        age_range: Tuple of (min_age, max_age) to filter by age
    
    Returns:
        DataFrame with treatment effects including value changes
    """
    # Filter medications
    meds_filtered = medications.filter(pl.col("drug").str.contains(f"(?i){drug_name}"))
    
    # Filter labs
    labs_filtered = labs.filter(pl.col("lab_test").str.contains(f"(?i){lab_test}"))
    
    # Apply demographic filters
    if sex_filter:
        meds_filtered = meds_filtered.filter(pl.col("sex") == sex_filter)
        labs_filtered = labs_filtered.filter(pl.col("sex") == sex_filter)
    
    if age_range:
        min_age, max_age = age_range
        meds_filtered = meds_filtered.filter(
            (pl.col("anchor_age") >= min_age) & (pl.col("anchor_age") <= max_age)
        )
        labs_filtered = labs_filtered.filter(
            (pl.col("anchor_age") >= min_age) & (pl.col("anchor_age") <= max_age)
        )
    
    print(f"Found {len(meds_filtered):,} medication administrations")
    print(f"Found {len(labs_filtered):,} lab measurements")
    
    if len(meds_filtered) == 0 or len(labs_filtered) == 0:
        print("Not enough data for analysis")
        return None, None
    
    # Join and calculate time differences
    combined = (
        meds_filtered.join(labs_filtered, on=["subject_id", "hadm_id"], how="inner", suffix="_lab")
        .with_columns([
            pl.col("charttime").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S"),
            pl.col("charttime_lab").str.strptime(pl.Datetime, "%Y-%m-%d %H:%M:%S")
        ])
        .with_columns([
            (pl.col("charttime_lab") - pl.col("charttime")).alias("time_diff_ns")
        ])
    )
    
    # Group columns
    group_cols = ["subject_id", "hadm_id", "drug", "charttime", "lab_test", "sex", "anchor_age"]
    
    # Find closest lab values before medication
    labs_before = (
        combined
        .filter(pl.col("time_diff_ns") < datetime.timedelta(0))
        .sort([*group_cols, "time_diff_ns"], descending=[False]*len(group_cols) + [True])
        .group_by(group_cols)
        .agg([
            pl.col("valuenum").first().alias("value_before"),
            pl.col("time_diff_ns").first().alias("time_before")
        ])
    )
    
    # Find closest lab values after medication
    labs_after = (
        combined
        .filter(pl.col("time_diff_ns") > datetime.timedelta(0))
        .sort([*group_cols, "time_diff_ns"])
        .group_by(group_cols)
        .agg([
            pl.col("valuenum").first().alias("value_after"),
            pl.col("time_diff_ns").first().alias("time_after")
        ])
    )
    
    # Join and calculate changes
    effects = (
        labs_before.join(labs_after, on=group_cols, how="inner")
        .with_columns([
            (pl.col("value_after") - pl.col("value_before")).alias("value_change"),
            ((pl.col("value_after") - pl.col("value_before")) / pl.col("value_before") * 100).alias("pct_change"),
            (pl.col("time_before").cast(pl.Duration).dt.total_hours() * -1).alias("hours_before"),
            (pl.col("time_after").cast(pl.Duration).dt.total_hours()).alias("hours_after")
        ])
    )
    
    print(f"\nFound {len(effects):,} medication-lab pairs with before/after measurements\n")
    
    if len(effects) == 0:
        return None, None
    
    # Generate summary statistics
    summary = (
        effects
        .group_by(["drug", "lab_test"])
        .agg([
            pl.count().alias("n_episodes"),
            pl.col("value_before").mean().alias("avg_value_before"),
            pl.col("value_after").mean().alias("avg_value_after"),
            pl.col("value_change").mean().alias("avg_change"),
            pl.col("pct_change").mean().alias("avg_pct_change"),
            pl.col("value_change").std().alias("std_change"),
            pl.col("hours_before").mean().alias("avg_hours_before"),
            pl.col("hours_after").mean().alias("avg_hours_after")
        ])
        .sort("n_episodes", descending=True)
    )
    
    return effects, summary

### Example: Analyze a specific drug and lab test

In [ ]:
# Example analysis - change these parameters
DRUG_NAME = "insulin"  # substring match, case-insensitive
LAB_TEST = "glucose"   # substring match, case-insensitive

effects, summary = analyze_treatment_effect(DRUG_NAME, LAB_TEST)

if summary is not None:
    print("Treatment Effectiveness Summary:")
    print(summary)

## 5. Analyze Treatment for Similar Patients

Filter by demographic characteristics similar to your target patient.

In [ ]:
# Analyze treatment effects for patients similar to our selected patient
patient_sex = patient_info['sex'][0] if len(patient_info) > 0 else 'M'
patient_age = patient_info['anchor_age'][0] if len(patient_info) > 0 else 50

# Define age range (±10 years)
age_range = (patient_age - 10, patient_age + 10)

print(f"Analyzing for patients similar to: {patient_sex}, age {patient_age} (range: {age_range[0]}-{age_range[1]})\n")

effects_similar, summary_similar = analyze_treatment_effect(
    DRUG_NAME, 
    LAB_TEST, 
    sex_filter=patient_sex, 
    age_range=age_range
)

if summary_similar is not None:
    print("\nTreatment Effectiveness for Similar Patients:")
    print(summary_similar)

## 6. Treatment Decision Support

View detailed effects to make treatment decisions.

In [ ]:
if effects_similar is not None:
    # Show distribution of effects
    print("Distribution of treatment effects:")
    print("\nValue changes (absolute):")
    print(effects_similar.select(["value_change"]).describe())
    
    print("\nPercentage changes:")
    print(effects_similar.select(["pct_change"]).describe())
    
    # Count positive vs negative effects
    positive_effects = len(effects_similar.filter(pl.col("value_change") < 0))  # Assuming decrease is good for glucose
    total_effects = len(effects_similar)
    
    print(f"\nTreatment outcomes:")
    print(f"  Positive outcomes (value decreased): {positive_effects} ({positive_effects/total_effects*100:.1f}%)")
    print(f"  Negative outcomes (value increased): {total_effects - positive_effects} ({(total_effects-positive_effects)/total_effects*100:.1f}%)")

## 7. View Individual Treatment Episodes

In [ ]:
if effects_similar is not None:
    # Show sample episodes
    print("Sample treatment episodes with before/after lab values:")
    sample_effects = (
        effects_similar
        .select([
            "subject_id", "sex", "anchor_age", "drug", "lab_test",
            "value_before", "value_after", "value_change", "pct_change",
            "hours_before", "hours_after"
        ])
        .head(10)
    )
    print(sample_effects)

## 8. Custom Query: Specific Patient History

Look at the selected patient's medication and lab history.

In [ ]:
# Get medication history for selected patient
patient_meds = (
    medications
    .filter(pl.col("subject_id") == PATIENT_ID)
    .filter(pl.col("drug").is_not_null())
    .select(["subject_id", "hadm_id", "charttime", "drug", "dose", "unit"])
    .sort("charttime")
)

print(f"Medication history for patient {PATIENT_ID}:")
print(patient_meds.head(20))
print(f"\nTotal medication records: {len(patient_meds)}")

In [ ]:
# Get lab history for selected patient
patient_labs = (
    labs
    .filter(pl.col("subject_id") == PATIENT_ID)
    .select(["subject_id", "hadm_id", "charttime", "lab_test", "valuenum", "valueuom"])
    .sort("charttime")
)

print(f"Lab history for patient {PATIENT_ID}:")
print(patient_labs.head(20))
print(f"\nTotal lab records: {len(patient_labs)}")

## 9. Decision Summary

Use the information above to make an informed treatment decision:

1. **Patient characteristics**: Check demographics (sex, age) of target patient
2. **Similar patient outcomes**: Review treatment effects for patients with similar demographics
3. **Success rate**: Check what percentage of treatments resulted in positive outcomes
4. **Effect magnitude**: Review average changes in lab values
5. **Patient history**: Consider the patient's previous medications and lab results

Based on these factors, you can decide whether to administer the treatment.